# Notebook 028: Offset Window Functions: lead, lag and Period-over-Period Churn

**Module:** Module 03: Advanced SQL, Aggregations & Analytical Window Functions  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 20: Finance Domain E2E Project & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
A SaaS subscription billing service analyzes customer MRR (Monthly Recurring Revenue). Finance requires month-over-month revenue variance, detection of plan upgrades/downgrades, and churn identification when an active account drops subscription renewal.

## 2. Technical Requirement & Problem Statement
Master offset window functions: implement `lag()` to retrieve previous month revenue and `lead()` to inspect subsequent renewal state. Calculate absolute and percentage period-over-period differences, and handle default fallback values on boundary rows.

## 3. Spark & Databricks Architecture Notes
Offset functions fetch values from neighboring rows within the same executor partition buffer without self-joining the table. Self-joins require a full Cartesian shuffle, whereas `lag()` operates in single-pass linear time $O(N)$ inside the partition.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_028_Offset_Window_Functions_lead,_lag_and_Period-over-Period_Churn")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating Monthly SaaS Subscription History
Create monthly subscription charges per customer account.

In [ ]:
subscription_history = [
    ("SUB-1", "2023-01", 100.0),
    ("SUB-1", "2023-02", 100.0),
    ("SUB-1", "2023-03", 150.0), # Upgrade
    ("SUB-1", "2023-04", 120.0), # Downgrade
    ("SUB-2", "2023-01", 50.0),
    ("SUB-2", "2023-02", 50.0),
    ("SUB-2", "2023-03", 0.0)    # Churn
]

schema = "sub_id STRING, billing_month STRING, mrr DOUBLE"
df_subs = spark.createDataFrame(subscription_history, schema)
df_subs.show()

## 6. Applying lag() and lead() with Default Values
Fetch previous month and next month MRR.

In [ ]:
window_sub = Window.partitionBy("sub_id").orderBy("billing_month")

df_lag_lead = df_subs.withColumn(
    "prev_mrr", F.lag("mrr", 1, default=0.0).over(window_sub)
).withColumn(
    "next_mrr", F.lead("mrr", 1, default=0.0).over(window_sub)
)

df_lag_lead.show()

## 7. Calculating Month-over-Month Growth & Change Classification
Compute variance and classify subscription status.

In [ ]:
df_variance = df_lag_lead.withColumn(
    "mrr_change", F.round(F.col("mrr") - F.col("prev_mrr"), 2)
).withColumn(
    "pct_change",
    F.when(F.col("prev_mrr") == 0, None)
     .otherwise(F.round(((F.col("mrr") - F.col("prev_mrr")) / F.col("prev_mrr")) * 100, 1))
).withColumn(
    "status_tag",
    F.when(F.col("prev_mrr") == 0, "NEW")
     .when(F.col("mrr") > F.col("prev_mrr"), "EXPANSION")
     .when((F.col("mrr") < F.col("prev_mrr")) & (F.col("mrr") > 0), "CONTRACTION")
     .when(F.col("mrr") == 0, "CHURNED")
     .otherwise("RETAINED")
)

df_variance.select("sub_id", "billing_month", "prev_mrr", "mrr", "mrr_change", "pct_change", "status_tag").show(truncate=False)

## 9. Key Architectural Takeaways
- **`lag(col, offset, default)` accesses preceding rows without requiring self-joins.**
- **`lead(col, offset, default)` peeks forward into subsequent rows in the partition.**
- **Always provide sensible `default` values to avoid propagating accidental nulls on boundary rows.**

## 10. Senior Data Engineering Interview Questions
1. Why is `lag()` significantly more efficient than joining a DataFrame to itself on `t.date = t_prev.date + 1`?
2. How do you calculate year-over-year growth for missing months using window functions?
3. What is the maximum offset value you can safely supply to `lag()` in PySpark?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
